# VARN Ask: train our own assistant model (LoRA)

Fine-tunes a small open model so the VARN assistant can run on our own weights. **This is the chat assistant, not the drone/aircraft detector.**

**Before you run it**
1. Right panel → **Session options**: set **Accelerator** to a GPU (T4 or P100) and turn **Internet** on.
2. Optional: if the base model asks for a Hugging Face login, add your token as a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets).
3. Click **Run All**. Expect about 1 to 2 hours. The adapter and a report appear in the **Output** tab.

It downloads the training data from GitHub (generated by `node training/build-dataset.mjs`), trains a rank-8 LoRA adapter, then checks it on held-out test cases. Nothing here touches your detector notebooks.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "bitsandbytes", "accelerate", "transformers"], check=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on an accelerator")

In [ ]:
import os, json, gzip, urllib.request, random, time

BASE = "mistralai/Mistral-7B-Instruct-v0.2"      # Apache-2.0. Same family as Cloudflare's @cf/mistral/mistral-7b-instruct-v0.2-lora
BRANCH = "feat/ask-panel"                        # change to "main" after the pull request is merged
RAW = f"https://raw.githubusercontent.com/bsalsa2/varn/{BRANCH}/platform/worker/training/data"
N_TRAIN = 3000                                   # examples used (of 4000); fewer is faster
MAX_LEN = 1536
EPOCHS = 1
OUT = "/kaggle/working"

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass                                         # fine if the base model needs no login

def fetch(name):
    with urllib.request.urlopen(f"{RAW}/{name}") as r:
        return gzip.decompress(r.read()).decode()

train = [json.loads(l) for l in fetch("train.jsonl.gz").splitlines() if l.strip()]
evalset = json.loads(fetch("eval.json.gz"))
random.Random(1).shuffle(train)
train = train[:N_TRAIN]
print(len(train), "training examples,", len(evalset), "held-out test cases")

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE, token=HF_TOKEN)
tok.pad_token = tok.eos_token

def fold(messages):
    """Mistral v0.2's chat template has no system role: put the system text at the start of the first user message."""
    if messages and messages[0]["role"] == "system":
        system, rest = messages[0]["content"], list(messages[1:])
        for i, m in enumerate(rest):
            if m["role"] == "user":
                rest[i] = {"role": "user", "content": system + "\n\n" + m["content"]}
                break
        return rest
    return list(messages)

def render(msgs, add_generation_prompt):
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=add_generation_prompt)

def encode(ex):
    """Token ids plus labels that cover ONLY the assistant turns after the question (history and prompts are masked out)."""
    msgs = fold(ex["messages"])
    first = ex["trainFrom"] - 1                  # one message shorter after folding the system text away
    full = render(msgs, False)
    spans = []
    for k, m in enumerate(msgs):
        if m["role"] == "assistant" and k >= first:
            spans.append((len(render(msgs[:k], True)), len(render(msgs[: k + 1], False))))
    enc = tok(full, add_special_tokens=False, return_offsets_mapping=True)
    labels = [tid if (e > s and any(a <= s and e <= b for a, b in spans)) else -100
              for (s, e), tid in zip(enc["offset_mapping"], enc["input_ids"])]
    return {"input_ids": enc["input_ids"], "labels": labels}

encoded = [encode(ex) for ex in train]
kept = [e for e in encoded if len(e["input_ids"]) <= MAX_LEN and any(l != -100 for l in e["labels"])]
print(f"{len(kept)} of {len(encoded)} examples fit in {MAX_LEN} tokens; longest {max(len(e['input_ids']) for e in encoded)}")
sample = kept[0]
print("TRAINED-ON TEXT (check this is only the assistant's JSON):")
print(tok.decode([t for t in sample["labels"] if t != -100]))

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={"": 0}, token=HF_TOKEN)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")
model = get_peft_model(model, lora)                # rank 8 keeps the adapter far below Cloudflare's size limit
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments

class Data(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    ids = [b["input_ids"] + [tok.pad_token_id] * (n - len(b["input_ids"])) for b in batch]
    lab = [b["labels"] + [-100] * (n - len(b["labels"])) for b in batch]
    att = [[1] * len(b["input_ids"]) + [0] * (n - len(b["input_ids"])) for b in batch]
    return {"input_ids": torch.tensor(ids), "labels": torch.tensor(lab), "attention_mask": torch.tensor(att)}

args = TrainingArguments(output_dir=f"{OUT}/run", per_device_train_batch_size=2, gradient_accumulation_steps=8, num_train_epochs=EPOCHS,
                         learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03, fp16=True, logging_steps=10,
                         save_strategy="no", report_to="none", gradient_checkpointing=True, optim="paged_adamw_8bit", remove_unused_columns=False)
t0 = time.time()
Trainer(model=model, args=args, train_dataset=Data(kept), data_collator=collate).train()
print(f"trained in {(time.time() - t0) / 60:.1f} min")
model.save_pretrained(f"{OUT}/adapter")
print(sorted(os.listdir(f"{OUT}/adapter")))

In [ ]:
def generate(msgs):
    text = render(fold(msgs), True)
    ids = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=400, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def parse(text):
    try:
        v = json.loads(text)
        return v if isinstance(v, dict) else None
    except Exception:
        i, j = text.find("{"), text.rfind("}")
        try:
            v = json.loads(text[i : j + 1])
            return v if isinstance(v, dict) else None
        except Exception:
            return None

def turn_ok(case, gold, got):
    """One assistant turn, given the correct conversation so far (the same tool results and prompts the Worker would send)."""
    g = parse(gold["content"])
    p = parse(got)
    if p is None or p.get("type") != g.get("type"):
        return False, p is not None
    if g["type"] == "tool_call":
        if p.get("name") != g["name"]:
            return False, True
        a, b = p.get("arguments") or {}, g.get("arguments") or {}
        return (a.get("commands") == b.get("commands") if g["name"] == "propose_command" else True), True
    exp = case["expect"]
    text = p.get("text", "")
    if "exact" in exp:
        return text == exp["exact"], True
    if sorted(p.get("sources", [])) != sorted(g.get("sources", [])):
        return False, True
    return all(s in text for s in exp.get("mustMention", [])), True

model.eval()
by_kind, valid, total_turns, passed = {}, 0, 0, 0
failures = []
for case in evalset:
    ok_case, ctx = True, list(case["prompt"])
    for gold in case["gold"]:
        if gold["role"] != "assistant":
            ctx.append(gold)
            continue
        got = generate(ctx)
        ok, is_json = turn_ok(case, gold, got)
        total_turns += 1; valid += is_json; ok_case = ok_case and ok
        if not ok:
            failures.append({"id": case["id"], "kind": case["kind"], "question": case["question"], "got": got[:300]})
        ctx.append(gold)                         # keep the gold turn so later turns see the right history
    k = by_kind.setdefault(case["kind"], [0, 0]); k[1] += 1; k[0] += ok_case; passed += ok_case

report = {"cases": len(evalset), "pass_rate": passed / len(evalset), "json_valid_rate": valid / total_turns,
          "by_kind": {k: f"{a}/{b}" for k, (a, b) in by_kind.items()}, "failures": failures[:25]}
json.dump(report, open(f"{OUT}/eval_report.json", "w"), indent=1)
print(json.dumps({k: v for k, v in report.items() if k != "failures"}, indent=1))
for f in failures[:8]:
    print("-", f["kind"], "|", f["question"], "->", f["got"][:160])

In [ ]:
import shutil
total = sum(os.path.getsize(f"{OUT}/adapter/{f}") for f in os.listdir(f"{OUT}/adapter"))
print(f"adapter size: {total / 1e6:.1f} MB (Cloudflare's limit for rank-8 adapters is about 100 MB)")
shutil.make_archive(f"{OUT}/varn-ask-adapter", "zip", f"{OUT}/adapter")
print("download varn-ask-adapter.zip and eval_report.json from the Output tab")

## Next
- Check `eval_report.json`. Compare `pass_rate` with the current model before using this adapter; it is only worth switching if it is at least as good.
- The adapter then needs uploading to Cloudflare Workers AI and the Worker setting `ASK_LORA` switched on (see `platform/worker/training/README.md`). Whether adapters work on the Free plan is still to be confirmed.
- This score is turn by turn with the correct earlier turns given. The full pipeline score (`training/score.mjs`) comes once the adapter is hosted.